# 10 — Video Inference: Vehicle Detection + Lane Segmentation

**Goal:** Run the trained joint model on a video from Google Drive and produce
an annotated output video with vehicle detections and lane overlays.

Input: `Drive/EcoCAR/video/*.mp4`
Output: Annotated video + per-frame stats

## 1 — Setup

In [ ]:
!pip install -q ultralytics opencv-python

In [ ]:
import os, sys, time, json, csv
import cv2
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')

ECOCAR_ROOT = '/content/drive/MyDrive/EcoCAR'
VIDEO_DIR = f'{ECOCAR_ROOT}/video'
WEIGHTS_DIR = f'{ECOCAR_ROOT}/weights'
OUTPUT_DIR = '/content/video_output'

os.makedirs(OUTPUT_DIR, exist_ok=True)

PROJECT_DIR = os.path.join(ECOCAR_ROOT, 'project')
assert os.path.isdir(PROJECT_DIR), f'Project not found at {PROJECT_DIR}'
sys.path.insert(0, PROJECT_DIR)

print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2 — Load Model

In [ ]:
from src.multitask_model import build_multitask_model
from src.utils.class_map import VEHICLE_CLASSES, NUM_VEHICLE_CLASSES

CKPT_PATH = os.path.join(WEIGHTS_DIR, 'vehicle_lane_v4_best.pt')
assert os.path.isfile(CKPT_PATH), f'Checkpoint not found: {CKPT_PATH}'

ckpt = torch.load(CKPT_PATH, map_location='cuda', weights_only=False)
arch = ckpt.get('arch_config', {})

model = build_multitask_model(
    weights='yolo26s.pt',
    lane_head_type=arch.get('lane_head_type', 'transformer'),
    lane_embed_dim=arch.get('lane_embed_dim', 128),
    lane_num_heads=arch.get('lane_num_heads', 4),
    lane_depth=arch.get('lane_depth', 2),
    mask_height=arch.get('mask_height', 160),
    mask_width=arch.get('mask_width', 160),
)
model.load_state_dict(ckpt['model_state_dict'], strict=False)
model = model.to('cuda').eval()
print('Model loaded.')

## 3 — Find Input Video

In [ ]:
video_files = []
if os.path.isdir(VIDEO_DIR):
    video_files = sorted([
        os.path.join(VIDEO_DIR, f) for f in os.listdir(VIDEO_DIR)
        if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))
    ])

if not video_files:
    print(f'No videos found in {VIDEO_DIR}')
    print('Upload a video to Drive/EcoCAR/video/ and re-run.')
else:
    for vf in video_files:
        sz = os.path.getsize(vf) / (1024**2)
        print(f'  {os.path.basename(vf)} ({sz:.1f} MB)')
    INPUT_VIDEO = video_files[0]
    print(f'\nUsing: {INPUT_VIDEO}')

## 4 — Run Video Inference

In [ ]:
BOX_COLORS = [
    (60, 180, 255), (100, 100, 255), (255, 220, 60),
    (100, 255, 100), (255, 100, 200),
]

try:
    from ultralytics.utils.nms import non_max_suppression
except ImportError:
    from ultralytics.utils.ops import non_max_suppression

IMG_SIZE = 640
CONF_THRESH = 0.3
LANE_THRESH = 0.5

cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out_name = os.path.splitext(os.path.basename(INPUT_VIDEO))[0] + '_annotated.mp4'
out_path = os.path.join(OUTPUT_DIR, out_name)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_path, fourcc, fps, (orig_w, orig_h))

frame_stats = []
frame_idx = 0

print(f'Input:  {INPUT_VIDEO}')
print(f'Output: {out_path}')
print(f'Resolution: {orig_w}x{orig_h} @ {fps:.1f} FPS, {total_frames} frames')
print(f'Processing...')

torch.cuda.synchronize()
t_start = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    t_frame_start = time.time()

    # Preprocess
    img_resized = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
    tensor = tensor.permute(2, 0, 1).unsqueeze(0).to('cuda')

    # Inference
    with torch.no_grad(), torch.amp.autocast('cuda'):
        out = model(tensor)

    # Decode detections
    det_out = out['det_output']
    if isinstance(det_out, (tuple, list)) and len(det_out) == 2:
        y_post = det_out[0]
        if isinstance(y_post, torch.Tensor) and y_post.dim() == 3 and y_post.shape[-1] == 6:
            valid = y_post[0][:, 4] > CONF_THRESH
            bboxes = y_post[0][valid].cpu().numpy()
        else:
            preds = non_max_suppression(y_post, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
            bboxes = preds[0].cpu().numpy()
    elif isinstance(det_out, torch.Tensor):
        preds = non_max_suppression(det_out, conf_thres=CONF_THRESH, iou_thres=0.45, max_det=100)
        bboxes = preds[0].cpu().numpy()
    else:
        bboxes = np.empty((0, 6))

    # Scale boxes back
    if len(bboxes):
        bboxes[:, [0, 2]] *= (orig_w / IMG_SIZE)
        bboxes[:, [1, 3]] *= (orig_h / IMG_SIZE)

    # Lane
    lane_prob = torch.sigmoid(out['lane_logits'])[0, 0].cpu().numpy()
    lane_mask = cv2.resize(lane_prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
    lane_binary = (lane_mask > LANE_THRESH).astype(np.uint8)

    # Draw
    vis = frame.copy()
    overlay = vis.copy()
    overlay[lane_binary == 1] = (255, 0, 255)
    vis = cv2.addWeighted(vis, 0.65, overlay, 0.35, 0)

    for box in bboxes:
        x1, y1, x2, y2, conf, cls = box
        cls_id = int(cls)
        if cls_id >= NUM_VEHICLE_CLASSES:
            continue
        color = BOX_COLORS[cls_id]
        label = f'{VEHICLE_CLASSES[cls_id]} {conf:.2f}'
        cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(vis, (int(x1), int(y1)-th-4), (int(x1)+tw, int(y1)), color, -1)
        cv2.putText(vis, label, (int(x1), int(y1)-2),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    writer.write(vis)

    torch.cuda.synchronize()
    t_frame_end = time.time()

    frame_stats.append({
        'frame': frame_idx,
        'time_ms': (t_frame_end - t_frame_start) * 1000,
        'n_detections': len(bboxes),
        'lane_coverage': float(lane_binary.mean()),
    })

    frame_idx += 1
    if frame_idx % 100 == 0:
        elapsed = time.time() - t_start
        fps_actual = frame_idx / elapsed
        print(f'  Frame {frame_idx}/{total_frames} ({fps_actual:.1f} FPS)')

cap.release()
writer.release()

total_time = time.time() - t_start
avg_fps = frame_idx / total_time

print(f'\nDone! {frame_idx} frames in {total_time:.1f}s ({avg_fps:.1f} FPS)')
print(f'Output: {out_path}')

## 5 — Frame Statistics

In [ ]:
import matplotlib.pyplot as plt

times = [s['time_ms'] for s in frame_stats]
dets = [s['n_detections'] for s in frame_stats]
lanes = [s['lane_coverage'] * 100 for s in frame_stats]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].plot(times, linewidth=0.5)
axes[0].set_title('Frame Latency (ms)')
axes[0].set_xlabel('Frame')
axes[0].axhline(np.mean(times), color='r', linestyle='--', label=f'Avg: {np.mean(times):.1f}ms')
axes[0].legend()

axes[1].plot(dets, linewidth=0.5)
axes[1].set_title('Detections per Frame')
axes[1].set_xlabel('Frame')

axes[2].plot(lanes, linewidth=0.5)
axes[2].set_title('Lane Coverage (%)')
axes[2].set_xlabel('Frame')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/frame_stats.png', dpi=150)
plt.show()

# Save stats CSV
csv_path = f'{OUTPUT_DIR}/frame_stats.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['frame', 'time_ms', 'n_detections', 'lane_coverage'])
    w.writeheader()
    w.writerows(frame_stats)
print(f'Frame stats saved: {csv_path}')

In [ ]:
# Copy output to Drive
import shutil
drive_out = f'{ECOCAR_ROOT}/video/output'
os.makedirs(drive_out, exist_ok=True)
shutil.copy2(out_path, drive_out)
shutil.copy2(csv_path, drive_out)
print(f'Results copied to {drive_out}')